# Open-SourceFine-Tuning

**Module 9 · Lesson 9.4**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The HuggingFace + TRL SFT Pipeline
- LoRA Config - Rank, Alpha, Target Modules
- QLoRA & VRAM - A 7B on a 16GB GPU
- Unsloth - 2x Faster, GGUF Export
- LoRA Variants - DoRA, rsLoRA, LoRA+, PiSSA
- Alignment - SFT → DPO → GRPO

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install transformers torch datasets "trl>=1.0" peft transformers datasets peft unsloth python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The A100 You Didn’t Need

> 💡 **Info**
>
> A team wanted to fine-tune a 7B model. Someone quoted full fine-tuning’s VRAM - **~120GB** - so they budgeted for a multi-GPU A100 rental and a week of setup. Then a junior engineer ran it on a **free Colab T4 (16GB)** the same afternoon. The trick was **QLoRA** (load the base in 4-bit, train only tiny LoRA adapters) plus **Unsloth** (custom kernels, ~70% less VRAM). The A100 was never needed. **Open-source fine-tuning is not just about control - QLoRA collapsed the hardware bar so far that a 7-8B model now fits on the free tier.** This lesson builds that pipeline: the current HuggingFace/TRL stack, the Unsloth fast path, LoRA variants, alignment, and how to serve the weights you own.

### 🎯 What you will be able to do after this lesson

- **Build** a QLoRA SFT pipeline on the current HuggingFace/TRL stack (BitsAndBytesConfig NF4 → LoraConfig all-linear + DoRA → SFTTrainer) and the Unsloth fast path.

- **Compare** the knobs that matter: LoRA rank/alpha/target-modules, the DoRA/rsLoRA variants, and VRAM by method (full vs LoRA vs QLoRA vs +Unsloth).

- **Operate** the alignment pipeline (SFT → DPO → GRPO) and choose a framework (Unsloth / Axolotl / LLaMA-Factory / torchtune).

- **Evaluate** deployment: merge vs vLLM multi-LoRA serving, GGUF/AWQ quantization, and the India open-weight path (Sarvam, IndiaAI GPUs).

> 📦 **Info**
>
> ✅ Before you startThis is the open-weight counterpart to **Lesson 9.3** (managed Vertex tuning): here you get the WEIGHTS and full control - and you run the GPUs. You should know the fine-tuning decision (9.1), the data format (9.2), and what a LoRA adapter is (9.1). The deep LoRA/QLoRA math is **Lesson 9.5**; this lesson is the practical tooling.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🔧 **Analogy**
>
> **Managed API fine-tuning is ordering a custom cake from a bakery** - you pick the flavors, they bake it, you cannot touch the oven. **Open-source fine-tuning is baking in your own kitchen** - every ingredient, every setting, every technique is yours, and you can make something the bakery never could. *Where the analogy breaks down:* your own kitchen means you also do the dishes - provisioning GPUs, quantizing, and serving are now YOUR job, not the platform’s. The control is real, and so is the responsibility.

> 📦 **Info**
>
> ⚠️ Two misconceptions this lesson kills**“Fine-tuning a 7B needs a data-center GPU.”** That 120GB figure is FULL fine-tuning. QLoRA (4-bit base + LoRA adapters) fits a 7-8B on a free 16GB T4; +Unsloth cuts it further. **“LoRA is lower quality than full fine-tuning.”** LoRA reaches ~95%+ of full quality on most tasks, and **DoRA can match or beat full fine-tuning** with zero inference overhead. You are trading almost nothing for a 10-20× cheaper run.

> 💡 **Info**
>
> 🚫 Anti-pattern: high rank on a few modulesThe instinct is “use a big rank on the attention layers.” The evidence says the opposite: a **moderate rank across ALL linear layers** (`target_modules="all-linear"` - q/k/v/o + gate/up/down) beats a high rank on q/v only, at a small extra VRAM cost. Spread the adapter wide, not deep.

---

## 🎯 Concept 1: The HuggingFace + TRL SFT Pipeline

### The HuggingFace + TRL SFT Pipeline

Load in 4-bit, attach LoRA, train with the CURRENT (TRL v1.0) API.

The open-source SFT pipeline is four moves: **load the base in 4-bit** (BitsAndBytesConfig NF4), **attach LoRA** to all linear layers (PEFT), **configure training** (SFTConfig), and **train** (SFTTrainer). The stack is HuggingFace [TRL](https://github.com/huggingface/trl) + PEFT + bitsandbytes. The catch that breaks last year’s code: **TRL v1.0 renamed the arguments** - `tokenizer=` became `processing_class=`, and `max_seq_length` became `max_length` (now on the SFTConfig, along with `dataset_text_field` and `report_to`).

> 🍳 **Analogy**
>
> The pipeline is an assembly line: raw model in one end, quantize it, bolt on the small adjustable part (LoRA), run it through the trainer, adapter out the other end. *Breaks down when:* the machine’s control panel changed labels - TRL v1.0 renamed the knobs, so code that reaches for the old `tokenizer=` / `max_seq_length` knobs grabs air and errors.

Your SFTTrainer code from last year passes `tokenizer=tokenizer` and `SFTConfig(max_seq_length=2048)`. You upgrade to TRL v1.0 and run it. What happens?

- `BitsAndBytesConfig(load_in_4bit, nf4, double_quant)` loads the frozen base in 4-bit = QLoRA.
- `get_peft_model(..., LoraConfig(target_modules="all-linear", use_dora=True, use_rslora=True))` attaches the adapter to every linear layer.
- `SFTTrainer(..., processing_class=tokenizer, args=SFTConfig(max_length=2048, report_to="wandb"))` is the CURRENT API - note the renamed args.

**📝 `01_pipeline.py HF + TRL`** - *v1.0*

> This snippet needs a GPU and packages not preinstalled here (peft/trl/bitsandbytes/unsloth), so it is shown for reference rather than run.

```python
# HUGGINGFACE + TRL SFT PIPELINE - the CURRENT (TRL v1.0) API.
# pip install transformers datasets peft trl bitsandbytes accelerate
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch

model_name = "google/gemma-3-4b-it"   # pick the current model: Gemma 3/4, Llama 3.3/4, Qwen3

# 1) load the base in 4-bit (QLoRA)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2) attach LoRA to ALL linear layers, DoRA + rsLoRA on (2026 defaults)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, target_modules="all-linear", lora_dropout=0.05,
    use_dora=True, use_rslora=True, task_type="CAUSAL_LM"))
model.print_trainable_parameters()    # ~0.5% trainable

# 3) train with the CURRENT TRL API (renamed args)
trainer = SFTTrainer(
    model=model,
    train_dataset=load_dataset("json", data_files="netsetos_train.jsonl")["train"],
    processing_class=tokenizer,        # was tokenizer=  (removed in TRL v1.0)
    args=SFTConfig(output_dir="./netsetos-lora", num_train_epochs=3,
                   per_device_train_batch_size=4, gradient_accumulation_steps=4,
                   learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.1,
                   max_length=2048, bf16=True, report_to="wandb"),   # was max_seq_length
)
# trainer.train(); model.save_pretrained("./netsetos-lora")   # ~tens of MB adapter

# Output: (illustrative - a real run needs a GPU)
# trainable params: 29,933,568 || all params: 4,300,000,000 || trainable%: 0.70
```

#### 💡 What just happened

⚡ What just happened? Four moves and a ~30M-parameter adapter (0.7% of a 4B model) comes out. The renamed args are the migration in a nutshell: **processing_class= (was tokenizer=)** and **max_length (was max_seq_length)** on the SFTConfig. The tradeoff of the open stack vs managed Vertex (9.3): you write and own every line here - more control, more surface area to keep current - but the result is a portable adapter file you can serve anywhere.

- Click each stage (load 4-bit, attach LoRA, configure, train, merge, quantize, serve) to see the code and the artifact it produces.
- See how QLoRA and the adapter file flow from one end to the other.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: LoRA Config - Rank, Alpha, Target Modules

### LoRA Config - Rank, Alpha, Target Modules

Why a tiny adapter works, and the config that captures years of research.

LoRA replaces a full weight update with two small matrices: `W’ = W₀ + (α/r)·BA`. For a `d×k` linear it trains `r·(d+k)` parameters instead of `d·k` - a fraction of a percent. The knobs: **rank r** (capacity), **alpha** (adaptation strength, set to 2×rank), and **target_modules** (which layers). The 2026 production default is `r=16, alpha=32, target_modules="all-linear", use_dora=True, use_rslora=True`. (The deep derivation is Lesson 9.5 - here we make it actionable.)

> 🎒 **Analogy**
>
> LoRA is sticky notes on a textbook instead of rewriting it. The book (base weights) stays; you add small annotations (A and B) that change how it reads. Rank r is how many notes per page; more notes = more nuance but more to manage. *Breaks down when:* you put a hundred notes on one page and none on the rest - high rank on a few modules. Better: a few notes on EVERY page (all-linear).

You compute LoRA’s trainable parameters for a 4096×4096 layer at rank 16. Roughly what fraction of the full layer is that?

- `lora_params(d, k, r) = r*(d+k)` is the size of the two adapter matrices A and B combined.
- The loop shows how trainable params (and the %) grow linearly with rank r.
- The last line scales it to a whole 7B across all-linear layers - still a fraction of a percent, so the saved adapter is tiny.

**📝 `02_lora.py LoRA`** - *math*

In [ ]:
# LoRA TRAINABLE PARAMETERS - a d x k linear gets two small matrices A (r x k), B (d x r).
# So it trains r*(d + k) params instead of d*k.
def lora_params(d, k, r):
    return r * (d + k)          # A: r*k  +  B: d*r
def dense_params(d, k):
    return d * k

d = k = 4096                     # a Gemma/Llama hidden dimension
for r in (8, 16, 32, 64):
    lp, dp = lora_params(d, k, r), dense_params(d, k)
    print(f"  r={r:<3} one 4096x4096 layer: {lp:>8,} trainable vs {dp:,} full  ({lp/dp*100:.2f}%)")

whole = lora_params(d, k, 16) * 7 * 32   # ~7 all-linear modules x ~32 layers
print(f"  r=16 all-linear across a 7B: ~{whole/1e6:.1f}M trainable (~{whole/7e9*100:.2f}% of 7B) -> a tiny adapter file")

# Output:
#   r=8   one 4096x4096 layer:   65,536 trainable vs 16,777,216 full  (0.39%)
#   r=16  one 4096x4096 layer:  131,072 trainable vs 16,777,216 full  (0.78%)
#   r=32  one 4096x4096 layer:  262,144 trainable vs 16,777,216 full  (1.56%)
#   r=64  one 4096x4096 layer:  524,288 trainable vs 16,777,216 full  (3.12%)
#   r=16 all-linear across a 7B: ~29.4M trainable (~0.42% of 7B) -> a tiny adapter file

#### 💡 What just happened

⚡ What just happened? At r=16 a 4096 layer trains ~131K params - 0.78% of the full 16.8M - and a whole 7B all-linear adapter is ~0.4% of the model. That is why LoRA files are tens of MB, not gigabytes. The tradeoff the rank sets: higher r = more capacity but more to train and a bigger overfit risk; the sweet spot is moderate rank spread across ALL linear layers, with alpha at 2×rank. The deep “why low-rank works” derivation is Lesson 9.5.

- Slide the rank, alpha, and hidden dimension and watch the trainable-parameter count and percentage move.
- See how the adapter stays a tiny fraction of the full model even at high rank.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: QLoRA & VRAM - A 7B on a 16GB GPU

### QLoRA & VRAM - A 7B on a 16GB GPU

4-bit NF4 quantization is the trick that collapsed the hardware bar.

This is the cold-open, quantified. Full fine-tuning holds the model plus gradients plus optimizer states - roughly 16 bytes per parameter, so a 7B needs ~120GB. **QLoRA** loads the frozen base in **4-bit NF4** (so the weights take ~4× less memory) and trains only the LoRA adapters - dropping the total to ~6–10GB. Add Unsloth’s gradient checkpointing and it fits comfortably on a free T4. That is why fine-tuning left the data center.

> 🧰 **Analogy**
>
> Full fine-tuning is moving into a mansion (every room heated). LoRA rents one adjustable room in an existing building. QLoRA also shrink-wraps the building to 4-bit so the whole thing weighs a quarter as much. *Breaks down when:* you need the full mansion - a deep capability change across all weights - then quantization noise and a frozen base are not enough, and full fine-tuning earns its VRAM.

Your 7B model needs ~120GB VRAM for full fine-tuning. How do you train it on a free 16GB T4?

- `vram(params_b, method)` gives rough training GB: full ~16 bytes/param, LoRA less, QLoRA (+Unsloth) least.
- `fits` returns the smallest GPU (T4/L4/A100) that holds it.
- The table is the cold-open in numbers: full needs multi-GPU; QLoRA (+Unsloth) lands on a free T4.

**📝 `03_vram.py`** - *VRAM*

In [ ]:
# QLoRA VRAM - how a 7B that needs 120GB for full fine-tuning fits a 16GB GPU.
def vram(params_b, method):
    return {"full": params_b*16, "lora": params_b*3.4, "qlora": params_b*1.0 + 3,
            "qlora+unsloth": params_b*0.7 + 2}[method]   # rough training GB

GPUS = [("T4", 16), ("L4", 24), ("A100-40", 40), ("A100-80", 80)]
def fits(v):
    for name, cap in GPUS:
        if v <= cap:
            return name
    return "multi-GPU"

print("  7B model, training VRAM by method + smallest GPU that fits:")
for m in ("full", "lora", "qlora", "qlora+unsloth"):
    v = vram(7, m)
    print(f"    {m:14s} ~{v:6.1f} GB -> {fits(v)}")
print("  QLoRA (+Unsloth) drops a 7B onto a free 16GB T4; full fine-tuning needs multi-GPU.")

# Output:
#   7B model, training VRAM by method + smallest GPU that fits:
#     full           ~ 112.0 GB -> multi-GPU
#     lora           ~  23.8 GB -> L4
#     qlora          ~  10.0 GB -> T4
#     qlora+unsloth  ~   6.9 GB -> T4
#   QLoRA (+Unsloth) drops a 7B onto a free 16GB T4; full fine-tuning needs multi-GPU.

#### 💡 What just happened

⚡ What just happened? Full 7B is ~112GB (multi-GPU); LoRA ~24GB (one L4); **QLoRA ~10GB and +Unsloth ~7GB - both fit a free 16GB T4.** QLoRA’s three tricks: NF4 quantization (information-optimal for normally-distributed weights), double quantization (quantize the quantization constants), and paged optimizers (spill to CPU RAM on a spike). The tradeoff: 4-bit introduces a little noise, but on most tasks it is invisible - and DoRA can recover the last bit of quality for free.

- Slide the model size and pick full / LoRA / QLoRA / +Unsloth, and watch the VRAM and which GPU it fits.
- Find the biggest model that still lands on a free 16GB T4.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Unsloth - 2x Faster, GGUF Export

### Unsloth - 2x Faster, GGUF Export

A drop-in speed layer, and one-call export to run your model locally.

Once the pipeline works, **Unsloth** ([github.com/unslothai/unsloth](https://github.com/unslothai/unsloth)) makes it faster and lighter without changing the recipe. `FastLanguageModel.from_pretrained` + `get_peft_model` replace the standard loaders; custom Triton kernels give ~2× speed and ~70% less VRAM, and `use_gradient_checkpointing="unsloth"` saves more. The killer feature for open-source: one-call export to **GGUF** (for Ollama / llama.cpp) or a merged model - so you can run your fine-tune locally in minutes.

> ⚡ **Analogy**
>
> Unsloth is a turbocharger you bolt onto the same engine. The car (TRL pipeline) drives the same; it just goes faster on less fuel (VRAM). *Breaks down when:* you need many engines at once - Unsloth’s free tier is single-GPU, so true multi-GPU scale is Axolotl’s job, not Unsloth’s.

Unsloth claims 2x speed and ~70% less VRAM without changing your training recipe. How is that possible?

- `FastLanguageModel.from_pretrained(load_in_4bit=True)` loads a pre-quantized base; Unsloth keeps its OWN `max_seq_length` arg (only TRL renamed it).
- `FastLanguageModel.get_peft_model(..., use_gradient_checkpointing="unsloth")` is the LoRA + memory trick.
- `save_pretrained_gguf` exports to GGUF (Q4_K_M) in one call - ready for Ollama.

**📝 `04_unsloth.py`** - *Unsloth*

> This snippet needs a GPU and packages not preinstalled here (peft/trl/bitsandbytes/unsloth), so it is shown for reference rather than run.

```python
# UNSLOTH - the same pipeline, ~2x faster and ~70% less VRAM (custom Triton kernels).
# pip install unsloth
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it-bnb-4bit",   # pre-quantized; pick the current model
    max_seq_length=2048, load_in_4bit=True)         # Unsloth keeps its own max_seq_length

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0, use_gradient_checkpointing="unsloth", use_dora=True)   # 60% less VRAM

# train with the same TRL SFTTrainer (processing_class; dataset_text_field on the SFTConfig)
# trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds,
#     args=SFTConfig(output_dir="./out", max_length=2048, dataset_text_field="text", bf16=True))
# trainer.train()

# one-call exports:
# model.save_pretrained_merged("merged", tokenizer)      # merged fp16
# model.save_pretrained_gguf("model.gguf", tokenizer)    # Q4_K_M for Ollama / llama.cpp

print("Unsloth: ~2x faster, ~70% less VRAM; fine-tunes a 7-8B on a free Colab T4; one-call GGUF export.")

# Output:
# Unsloth: ~2x faster, ~70% less VRAM; fine-tunes a 7-8B on a free Colab T4; one-call GGUF export.
```

#### 💡 What just happened

⚡ What just happened? Same SFT recipe, but ~2× faster and ~70% less VRAM via custom kernels - enough to fine-tune a 7-8B on a free Colab T4 that would otherwise want an A100. And `save_pretrained_gguf` is the open-source payoff: one call turns your adapter into a GGUF file you run locally on Ollama, no cloud needed. The tradeoff: the free tier is single-GPU, so multi-GPU production runs move to Axolotl (step 7).

---

## 🎯 Concept 5: LoRA Variants - DoRA, rsLoRA, LoRA+, PiSSA

### LoRA Variants - DoRA, rsLoRA, LoRA+, PiSSA

Single-flag upgrades that add quality or stability for free.

Plain LoRA is the baseline; four variants improve it, each a single flag. **DoRA** (decompose the weight into magnitude + direction, LoRA the direction) adds ~1–3% quality with zero inference overhead - turn it on by default. **rsLoRA** (scale by α/√r) keeps gradients stable at high rank - essential for r≥32. **LoRA+** (higher LR on B) is experimental and conflicts with rsLoRA. **PiSSA** (SVD init) helps SFT but fails in GRPO/RL.

> ⚙️ **Analogy**
>
> The variants are tuning trims on the same engine. DoRA is a better spark plug (small, free power). rsLoRA is a governor that keeps the engine stable when you rev it high (large rank). *Breaks down when:* you fit trims that fight each other - LoRA+ and rsLoRA conflict - or use a trim outside its regime, like PiSSA in reinforcement learning where it fails.

You’re tuning at a high rank (r=64) and the training loss is unstable / spiking. Which variant flag fixes it?

- `recommend_variant(rank, goal)` always turns on DoRA (the free upgrade).
- At rank ≥32 it adds rsLoRA for gradient stability.
- It warns when a variant is out of its lane: PiSSA fails in reasoning/GRPO, and LoRA+ conflicts with rsLoRA.

**📝 `05_variant.py`** - *Variants*

In [ ]:
# LoRA VARIANTS - which single-flag upgrade to turn on, and when.
def recommend_variant(rank, goal):
    flags = {"use_dora": True}          # DoRA is a free upgrade in 2026 - default on
    notes = ["use_dora=True  (+1-3% quality, zero inference overhead)"]
    if rank >= 32:
        flags["use_rslora"] = True
        notes.append("use_rslora=True  (stable gradients - essential at rank >= 32)")
    if goal == "reasoning":
        notes.append("PiSSA helps SFT but FAILS in GRPO/RL - do not use it for reasoning")
    if goal == "speed-experiment":
        notes.append("LoRA+ (higher LR on B) is experimental and conflicts with rsLoRA")
    return flags, notes

for rank, goal in [(16, "style"), (64, "domain"), (16, "reasoning")]:
    flags, notes = recommend_variant(rank, goal)
    print(f"  rank={rank:<3} goal={goal:16s} -> {flags}")
    for note in notes:
        print(f"      - {note}")

# Output:
#   rank=16  goal=style            -> {'use_dora': True}
#       - use_dora=True  (+1-3% quality, zero inference overhead)
#   rank=64  goal=domain           -> {'use_dora': True, 'use_rslora': True}
#       - use_dora=True  (+1-3% quality, zero inference overhead)
#       - use_rslora=True  (stable gradients - essential at rank >= 32)
#   rank=16  goal=reasoning        -> {'use_dora': True}
#       - use_dora=True  (+1-3% quality, zero inference overhead)
#       - PiSSA helps SFT but FAILS in GRPO/RL - do not use it for reasoning

#### 💡 What just happened

⚡ What just happened? The recommender encodes the 2026 defaults: **DoRA always, rsLoRA at r≥32**, and stay away from PiSSA for reasoning and LoRA+ with rsLoRA. DoRA is the standout - it beat full fine-tuning on math reasoning (a reported ~46.6% vs ~44.9%) with zero inference cost because magnitude and direction merge back into the base. The tradeoff is essentially none: these are single-flag, near-free upgrades, which is why they are the default, not the exception.

- Toggle your scenario (rank, goal: style / high-rank domain / reasoning) and see which variant flags light up.
- See the guardrails: PiSSA off for reasoning, LoRA+ conflicts with rsLoRA.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Alignment - SFT → DPO → GRPO

### Alignment - SFT → DPO → GRPO

Three trainers, one order: foundation, then preferences, then verifiable reasoning.

SFT teaches the behavioral foundation; two alignment methods refine it. **DPO** learns from preference pairs (chosen vs rejected) with no reward model - and with LoRA, TRL cleverly disables the adapter to compute the reference logits, so no separate model is loaded. **GRPO** samples several completions per prompt and rewards the verifiably-correct ones (no reward model either) - it powers DeepSeek-R1-style reasoning. TRL v1.0 made SFT, DPO, and GRPO the **stable core**; ORPO moved to `trl.experimental`.

> 🎓 **Analogy**
>
> It is a coaching progression. SFT is teaching the fundamentals by demonstration. DPO is showing pairs and saying “this answer is better than that one” - for subjective quality like tone. GRPO is letting the student attempt a problem many ways and rewarding the runs that are verifiably correct - for math/code. *Breaks down when:* you use the wrong coach for the skill - GRPO needs a checkable answer, so it cannot grade ‘warmer tone’; that is DPO’s job.

You need to fine-tune for BOTH customer-support style AND math reasoning. Which methods?

- `pick_alignment(goal, data)` maps goal-and-data to the method: format+pairs→SFT, style+preferences→DPO, reasoning+verifiable→GRPO.
- It notes DPO’s LoRA trick (disable the adapter for reference logits) and GRPO’s in-group reward normalization.
- The closing line is the standard order: SFT → DPO → GRPO, each building on the last.

**📝 `06_align.py`** - *Alignment*

In [ ]:
# ALIGNMENT - match the method to the goal + the data (SFT -> DPO -> GRPO).
def pick_alignment(goal, data):
    table = {
        ("format",    "pairs"):       ("SFT",  "input-output pairs teach the behavioral foundation - always first"),
        ("style",     "preferences"): ("DPO",  "chosen/rejected pairs; no reward model (TRL disables the adapter for ref logits)"),
        ("reasoning", "verifiable"):  ("GRPO", "sample G completions, reward correct answers, normalize in-group; powers DeepSeek-R1"),
        ("memory",    "preferences"): ("ORPO", "single-stage SFT+alignment, no reference model (trl.experimental in v1.0)"),
    }
    return table.get((goal, data), ("SFT", "default: start with supervised fine-tuning"))

for goal, data in [("format","pairs"), ("style","preferences"), ("reasoning","verifiable"), ("memory","preferences")]:
    method, why = pick_alignment(goal, data)
    print(f"  goal={goal:10s} data={data:12s} -> {method:5s} | {why}")
print("  Standard pipeline: SFT (foundation) -> DPO (subjective quality) -> GRPO (verifiable reasoning).")

# Output:
#   goal=format     data=pairs        -> SFT   | input-output pairs teach the behavioral foundation - always first
#   goal=style      data=preferences  -> DPO   | chosen/rejected pairs; no reward model (TRL disables the adapter for ref logits)
#   goal=reasoning  data=verifiable   -> GRPO  | sample G completions, reward correct answers, normalize in-group; powers DeepSeek-R1
#   goal=memory     data=preferences  -> ORPO  | single-stage SFT+alignment, no reference model (trl.experimental in v1.0)
#   Standard pipeline: SFT (foundation) -> DPO (subjective quality) -> GRPO (verifiable reasoning).

#### 💡 What just happened

⚡ What just happened? The selector encodes the pipeline: **SFT (foundation) → DPO (subjective quality) → GRPO (verifiable reasoning)**, all stable-core in TRL v1.0. DPO’s memory trick (disable the LoRA adapter to get reference logits) means no second model; GRPO needs only a reward function (is the answer correct?), no trained reward model. The tradeoff: each stage adds a data requirement (pairs for DPO, a verifier for GRPO), so you climb only as far as your goal - and your data - support.

- Toggle your goal (teach a format, shift tone, improve reasoning) and the data you have, and watch the method light up.
- See why GRPO needs a verifiable reward and DPO needs preference pairs.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Frameworks + Serving - and Multi-LoRA

### Frameworks + Serving - and Multi-LoRA

Choose the trainer for your hardware, then serve one base with many adapters.

Four frameworks cover the field: **Unsloth+TRL** (single-GPU speed), **Axolotl** (production multi-GPU, YAML, FSDP2/DeepSpeed), **LLaMA-Factory** (zero-code web UI), **torchtune** (pure PyTorch control). For serving: **merge** the adapter for a single-task model (zero overhead), but keep adapters separate for the production win - **vLLM multi-LoRA** serves ONE base model plus many small adapters, routed per request. Quantize to **GGUF Q4_K_M** for local (Ollama) or **AWQ** for GPU (via llm-compressor; AutoAWQ is archived).

> 🏪 **Analogy**
>
> Multi-LoRA serving is a food court with one big kitchen (the base model) and many small counters (adapters). You do not build a separate restaurant per cuisine - each order just tells the kitchen which counter’s recipe to use. *Breaks down when:* two counters need totally different kitchens (different base models) - then multi-LoRA cannot share, and you are back to separate deployments.

You trained 3 task-specific LoRA adapters (support, code, legal) on the same base. How do you serve all three most efficiently?

- `pick_framework` routes on hardware/needs: multi-GPU→Axolotl, zero-code→LLaMA-Factory, max-control→torchtune, else Unsloth+TRL.
- `pick_serving` returns merge for one task, and vLLM multi-LoRA for many.
- The multi-LoRA line is the key production pattern: one base, N adapters, per-request routing.

**📝 `07_serve.py`** - *Frameworks*

In [ ]:
# FRAMEWORK + SERVING - pick the trainer, then the serving pattern.
def pick_framework(multi_gpu, want_zero_code, want_max_control):
    if multi_gpu:        return ("Axolotl", "production multi-GPU via YAML + FSDP2/DeepSpeed")
    if want_zero_code:   return ("LLaMA-Factory", "zero-code web UI, 100+ models (Unsloth backend)")
    if want_max_control: return ("torchtune", "pure PyTorch, maximum control")
    return ("Unsloth + TRL", "single-GPU speed: ~2x faster, ~70% less VRAM")

def pick_serving(n_tasks):
    if n_tasks == 1:
        return "merge_and_unload -> one merged model (zero adapter overhead)"
    return f"vLLM multi-LoRA: ONE base + {n_tasks} adapters (--enable-lora --lora-modules), swapped per request"

for label, args in [("Single GPU, code", (False, False, False)),
                    ("4x A100 cluster", (True, False, False)),
                    ("Analyst, no code", (False, True, False))]:
    fw, why = pick_framework(*args)
    print(f"  {label:20s} -> {fw:15s} | {why}")
print(f"  Serving 1 task : {pick_serving(1)}")
print(f"  Serving 3 tasks: {pick_serving(3)}")

# Output:
#   Single GPU, code     -> Unsloth + TRL   | single-GPU speed: ~2x faster, ~70% less VRAM
#   4x A100 cluster      -> Axolotl         | production multi-GPU via YAML + FSDP2/DeepSpeed
#   Analyst, no code     -> LLaMA-Factory   | zero-code web UI, 100+ models (Unsloth backend)
#   Serving 1 task : merge_and_unload -> one merged model (zero adapter overhead)
#   Serving 3 tasks: vLLM multi-LoRA: ONE base + 3 adapters (--enable-lora --lora-modules), swapped per request

#### 💡 What just happened

⚡ What just happened? The framework follows the hardware (Unsloth single-GPU, Axolotl multi-GPU, LLaMA-Factory no-code, torchtune raw control), and serving follows the task count: **merge for one task, vLLM multi-LoRA for many**. Multi-LoRA is the efficiency win - one base in memory plus tiny adapters swapped per request, instead of N full model copies. The tradeoff: multi-LoRA only shares adapters trained on the SAME base, so a mix of base models still means separate deployments.

- Slide the number of task adapters and compare memory: multi-LoRA (one base + adapters) vs one full model copy per task.
- See the memory gap widen as you add adapters.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: India Open-Source Fine-Tuning

### India Open-Source Fine-Tuning

Sarvam models, IndiaAI GPUs, and a budget path from ₹0 up.

Open weights make India’s fine-tuning story cheap and sovereign. **Sarvam** ships open models (Sarvam-1 2B, Sarvam-M 24B, Apache) with an Indic-first tokenizer that costs ~half the tokens of Llama on Hindi. Compute is the cheapest globally: **IndiaAI Mission** GPUs at roughly ₹65–92/GPU-hr, **Jarvislabs** A100 ~₹108/hr, plus Indian clouds. The path scales with budget: free (Colab + Unsloth + Sarvam-1) → Jarvislabs → IndiaAI for production.

> 🇮🇳 **Analogy**
>
> India’s stack is a well-stocked community kitchen: free burners for beginners (Colab), affordable rented stations (Jarvislabs), and an industrial subsidized kitchen for scale (IndiaAI). And the ingredients are pre-portioned for the local cuisine - Sarvam’s Indic tokenizer means the same Hindi dish costs half the tokens. *Breaks down when:* your task is English/code-dominated - there the Indic tokenizer’s advantage shrinks and a general model may serve as well.

You want to fine-tune a Hindi chatbot on a ₹500 budget. Best approach?

- `india_plan(budget_inr)` maps rupees to a concrete stack: free Colab, then Jarvislabs, then subsidized IndiaAI GPUs.
- Each tier names the Sarvam model that fits and the deploy path (GGUF/Ollama for local).
- It divides the budget by the Jarvislabs hourly rate to show how many A100 hours you get.

**📝 `08_india.py`** - *India*

In [ ]:
# INDIA - budget decides the stack; Sarvam + Unsloth + IndiaAI is the cheap path.
def india_plan(budget_inr):
    if budget_inr < 100:
        return ("free Colab T4 + Unsloth", "Sarvam-1 (2B) or Gemma 3 4B, QLoRA", "export GGUF -> Ollama (local, free)")
    if budget_inr <= 600:
        hrs = round(budget_inr / 108, 1)   # Jarvislabs A100 ~ Rs 108/hr
        return (f"Jarvislabs A100 (~Rs 108/hr, ~{hrs} hrs)", "Sarvam-1 / Sarvam-M (24B) QLoRA", "IndicAlign data (free) + GGUF export")
    return ("IndiaAI Mission GPUs (~Rs 65-92/hr, subsidized)", "Sarvam-M 24B QLoRA", "production serving on Indian cloud")

for b in (0, 500, 5000):
    stack, model, deploy = india_plan(b)
    print(f"  budget Rs {b:<5} -> {stack}")
    print(f"      model: {model}   deploy: {deploy}")
print("  Sarvam-1's Indic tokenizer costs ~half the tokens of Llama on Hindi -> cheaper training + inference.")

# Output:
#   budget Rs 0     -> free Colab T4 + Unsloth
#       model: Sarvam-1 (2B) or Gemma 3 4B, QLoRA   deploy: export GGUF -> Ollama (local, free)
#   budget Rs 500   -> Jarvislabs A100 (~Rs 108/hr, ~4.6 hrs)
#       model: Sarvam-1 / Sarvam-M (24B) QLoRA   deploy: IndicAlign data (free) + GGUF export
#   budget Rs 5000  -> IndiaAI Mission GPUs (~Rs 65-92/hr, subsidized)
#       model: Sarvam-M 24B QLoRA   deploy: production serving on Indian cloud
#   Sarvam-1's Indic tokenizer costs ~half the tokens of Llama on Hindi -> cheaper training + inference.

> 📦 **Info**
>
> 🇮🇳 The India open-weight stack (Jul 2026)
> - **Models:** Sarvam-1 (2B, 10 Indic langs, efficient tokenizer), Sarvam-M (24B), Sarvam-M 24B (Apache); Krutrim-2 (~12B, verify current release). Gemma 3 and Llama also fine-tune well.
> - **Data:** AI4Bharat IndicAlign (tens of millions of instruction pairs), IndicCorp v2 (CC-0), IndicTrans2 to augment English datasets.
> - **Compute:** IndiaAI Mission (~₹65–92/GPU-hr, subsidized) is among the cheapest globally; Jarvislabs A100 ~₹108/hr; E2E / Yotta Indian clouds.
> - **Budget path:** ₹0 (Colab T4 + Unsloth + Sarvam-1) → ₹500 (Jarvislabs A100 for a few hours) → production on IndiaAI.
> - **Compliance:** open weights + Indian compute keep data in-country; DPDP still applies to personal training data (up to ₹250 crore).

#### 💡 What just happened

⚡ What just happened? India’s open-weight path is cheap AND sovereign: a Sarvam base (Indic tokenizer → ~half the tokens on Hindi), free IndicAlign data, and IndiaAI/Jarvislabs GPUs at a fraction of hyperscaler prices - all keeping data in-country. The tradeoff that decides your base is fertility over familiarity (from 9.2): a smaller Indic-first model beats a bigger generic one on Hindi. Start on a free T4 and scale only when the results justify it.

## Putting It Together

> ✅ **Info**
>
> 🧠 The open-source fine-tuning path
> - **Load + adapt.** 4-bit base (QLoRA) + LoRA on all-linear with DoRA/rsLoRA, on the current TRL v1.0 API (`processing_class`, `max_length`).
> - **Fit it cheaply.** QLoRA + Unsloth drops a 7-8B onto a free T4; export GGUF to run it locally.
> - **Align it.** SFT → DPO (preferences) → GRPO (verifiable reasoning), the TRL stable core.
> - **Pick the tool.** Unsloth (single-GPU) / Axolotl (multi-GPU) / LLaMA-Factory (no-code) / torchtune (control).
> - **Serve the weights you own.** Merge for one task; vLLM multi-LoRA for many; GGUF/AWQ to quantize. India: Sarvam + IndiaAI.
> The through-line: **the open stack trades convenience for control and cost - you own the weights and run the GPUs, and QLoRA makes the GPU a free T4.**

> 📦 **Info**
>
> 🔗 Where this goes next
> - In Lesson 9.5 we’ll come back to the **LoRA & QLoRA** mechanics under the hood - the derivation this lesson used at a practical level.
> - Lesson 9.6 picks this up for **evaluation, merging, and deployment** beyond the overview here.
> - Later, in Module 14 (**LLMOps**), we’ll return to serving and monitoring your open-weight model in production.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Open-SourceFine-Tuning**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-9_4.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-9.4.html` - regenerate this notebook whenever the source HTML is updated.*
